In [11]:
"""
Build LSTM sequences for coral bleaching prediction.

For each bleaching event in the GCBD, extracts a 52-week lookback 
sequence of thermal stress features from local CoRTAD v6 NetCDF files.

Inputs:
  - cortadv6_FilledSST.nc  (50 GB)  -> FilledSST
  - cortadv6_SSTA.nc       (111 GB) -> SSTA, SSTA_DHW, SSTA_Frequency
  - cortadv6_TSA.nc        (83 GB)  -> TSA, TSA_DHW, TSA_Frequency
  - global_bleaching_environmental.csv (GCBD)

Outputs:
  - sequences.npz: X (N, 52, 7), y (N,), metadata (N, 4)
"""

import os
import xarray as xr
import pandas as pd
import numpy as np
import time

# ──────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────
LOOKBACK_WEEKS = 52  # 1-year lookback
GCBD_PATH = "datasets/global_bleaching_environmental.csv"
SST_PATH = "datasets/cortadv6_FilledSST.nc"
SSTA_PATH = "datasets/cortadv6_SSTA.nc"
TSA_PATH = "datasets/cortadv6_TSA.nc"
OUTPUT_PATH = "datasets/sequences.npz"

# Bleaching severity bins: 0=none, 1=low, 2=moderate, 3=severe
BLEACH_BINS = [-1, 1, 10, 50, 100]
BLEACH_LABELS = [0, 1, 2, 3]

# Features to extract (7 total)
FEATURE_NAMES = [
    "FilledSST",      # Raw SST
    "SSTA",           # SST Anomaly (vs weekly climatology)
    "SSTA_DHW",       # SSTA-based Degree Heating Weeks
    "SSTA_Frequency", # SSTA frequency (times SSTA>=1 in past 52 weeks)
    "TSA",            # Thermal Stress Anomaly (vs max monthly mean)
    "TSA_DHW",        # TSA-based Degree Heating Weeks (standard DHW)
    "TSA_Frequency",  # TSA frequency (times TSA>=1 in past 52 weeks)
]

# ──────────────────────────────────────────────────────────────
# STEP 1: Load GCBD and prepare events
# ──────────────────────────────────────────────────────────────
print("=" * 70)
print("STEP 1: Loading GCBD")
print("=" * 70)

gcbd = pd.read_csv(GCBD_PATH)
print(f"  Total records: {len(gcbd)}")

# Filter to records that have enough info for a label and a date
gcbd = gcbd.dropna(subset=["Percent_Bleaching", "Date_Year", "Latitude_Degrees", "Longitude_Degrees"])
gcbd = gcbd[gcbd["Date_Year"] >= 1983]  # Need 1 year of lookback from 1982 start
print(f"  After filtering: {len(gcbd)}")

# Coerce non-numeric Percent_Bleaching entries to NaN
gcbd["Percent_Bleaching"] = pd.to_numeric(gcbd["Percent_Bleaching"], errors="coerce")

STEP 1: Loading GCBD
  Total records: 41361
  After filtering: 41359


/var/folders/1x/6tp2zz4x70q7nl89j3cmqf7c0000gn/T/ipykernel_3867/755801209.py:55: DtypeWarning: Columns (0: Distance_to_Shore, 1: Turbidity, 2: Percent_Bleaching) have mixed types. Specify dtype option on import or set low_memory=False.
  gcbd = pd.read_csv(GCBD_PATH)


In [13]:
# Drop rows where Percent_Bleaching couldn't be parsed to a number
gcbd = gcbd.dropna(subset=["Percent_Bleaching"])

# Create ordinal bleaching class
# Clip to bin range to avoid NaN from pd.cut on out-of-range values
gcbd["bleach_class"] = pd.cut(
    gcbd["Percent_Bleaching"].clip(0, 100),
    bins=BLEACH_BINS,
    labels=BLEACH_LABELS,
    include_lowest=True,
).astype(int)

# Create approximate date (use middle of month, default to June if month missing)
gcbd["event_month"] = gcbd["Date_Month"].fillna(6).astype(int).clip(1, 12)
gcbd["event_date"] = pd.to_datetime(
    gcbd["Date_Year"].astype(int).astype(str) + "-" +
    gcbd["event_month"].astype(str).str.zfill(2) + "-15"
)

print(f"  After dropping non-numeric bleaching values: {len(gcbd)}")
print(f"  Date range: {gcbd['event_date'].min()} to {gcbd['event_date'].max()}")
print(f"  Class distribution:")
for c in BLEACH_LABELS:
    n = (gcbd["bleach_class"] == c).sum()
    print(f"    Class {c}: {n} ({n/len(gcbd)*100:.1f}%)")

# ──────────────────────────────────────────────────────────────
# STEP 2: Open CoRTAD files and extract unique site time series
# ──────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 2: Opening local CoRTAD files")
print("=" * 70)

t0 = time.time()

# Use chunks to avoid loading everything into memory
sst_ds = xr.open_dataset(SST_PATH, chunks={"time": 104})
ssta_ds = xr.open_dataset(SSTA_PATH, chunks={"time": 104})
tsa_ds = xr.open_dataset(TSA_PATH, chunks={"time": 104})

print(f"  Opened in {time.time()-t0:.1f}s")

# xarray already decodes CoRTAD time into datetime64 values
dates = pd.DatetimeIndex(sst_ds["time"].values)
print(f"  Time range: {dates[0]} to {dates[-1]} ({len(dates)} weeks)")

# ──────────────────────────────────────────────────────────────
# STEP 3: Extract unique site locations
# ──────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 3: Extracting unique site time series")
print("=" * 70)

unique_sites = gcbd[["Latitude_Degrees", "Longitude_Degrees"]].drop_duplicates().reset_index(drop=True)
print(f"  Unique sites: {len(unique_sites)}")

lats = xr.DataArray(unique_sites["Latitude_Degrees"].values, dims="site")
lons = xr.DataArray(unique_sites["Longitude_Degrees"].values, dims="site")

# Extract all sites at once for each variable
# This is the slow part — reads from disk but only the needed grid cells
print("  Extracting FilledSST...", end=" ", flush=True)
t0 = time.time()
sst_all = sst_ds["FilledSST"].sel(lat=lats, lon=lons, method="nearest").load()
print(f"done ({time.time()-t0:.1f}s)")

print("  Extracting SSTA...", end=" ", flush=True)
t0 = time.time()
ssta_all = ssta_ds["SSTA"].sel(lat=lats, lon=lons, method="nearest").load()
print(f"done ({time.time()-t0:.1f}s)")

print("  Extracting SSTA_DHW...", end=" ", flush=True)
t0 = time.time()
ssta_dhw_all = ssta_ds["SSTA_DHW"].sel(lat=lats, lon=lons, method="nearest").load()
print(f"done ({time.time()-t0:.1f}s)")

print("  Extracting SSTA_Frequency...", end=" ", flush=True)
t0 = time.time()
ssta_freq_all = ssta_ds["SSTA_Frequency"].sel(lat=lats, lon=lons, method="nearest").load()
print(f"done ({time.time()-t0:.1f}s)")

print("  Extracting TSA...", end=" ", flush=True)
t0 = time.time()
tsa_all = tsa_ds["TSA"].sel(lat=lats, lon=lons, method="nearest").load()
print(f"done ({time.time()-t0:.1f}s)")

print("  Extracting TSA_DHW...", end=" ", flush=True)
t0 = time.time()
tsa_dhw_all = tsa_ds["TSA_DHW"].sel(lat=lats, lon=lons, method="nearest").load()
print(f"done ({time.time()-t0:.1f}s)")

print("  Extracting TSA_Frequency...", end=" ", flush=True)
t0 = time.time()
tsa_freq_all = tsa_ds["TSA_Frequency"].sel(lat=lats, lon=lons, method="nearest").load()
print(f"done ({time.time()-t0:.1f}s)")

# Stack into a single array: (time, site, features)
all_features = np.stack([
    sst_all.values,
    ssta_all.values,
    ssta_dhw_all.values,
    ssta_freq_all.values,
    tsa_all.values,
    tsa_dhw_all.values,
    tsa_freq_all.values,
], axis=-1)  # shape: (num_weeks, num_sites, 7)

print(f"\n  Combined feature array shape: {all_features.shape}")
print(f"  (time_steps, sites, features)")

# Build a lookup: (lat, lon) -> site index
site_lookup = {}
for idx, row in unique_sites.iterrows():
    key = (round(row["Latitude_Degrees"], 4), round(row["Longitude_Degrees"], 4))
    site_lookup[key] = idx

# ──────────────────────────────────────────────────────────────
# STEP 4: Build sequences for each bleaching event
# ──────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 4: Building sequences")
print("=" * 70)

X_list = []
y_list = []
meta_list = []  # lat, lon, year, month
skipped = 0

for i, row in gcbd.iterrows():
    lat = row["Latitude_Degrees"]
    lon = row["Longitude_Degrees"]
    event_date = row["event_date"]
    label = row["bleach_class"]
    
    # Find site index
    key = (round(lat, 4), round(lon, 4))
    site_idx = site_lookup.get(key)
    if site_idx is None:
        skipped += 1
        continue
    
    # Find the time index closest to the event date
    end_idx = int(np.argmin(np.abs(dates - event_date)))
    start_idx = end_idx - LOOKBACK_WEEKS
    
    # Check bounds
    if start_idx < 0:
        skipped += 1
        continue
    
    # Extract sequence: (52, 7)
    seq = all_features[start_idx:end_idx, site_idx, :]
    
    if seq.shape[0] != LOOKBACK_WEEKS:
        skipped += 1
        continue
    
    # Check for all-NaN sequences (land pixels etc.)
    if np.all(np.isnan(seq)):
        skipped += 1
        continue
    
    X_list.append(seq)
    y_list.append(label)
    meta_list.append([lat, lon, row["Date_Year"], row["event_month"]])
    
    if len(X_list) % 5000 == 0:
        print(f"  Built {len(X_list)} sequences ({skipped} skipped)...")

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.int32)
meta = np.array(meta_list, dtype=np.float32)

print(f"\n  Final dataset:")
print(f"    X shape: {X.shape}  (samples, timesteps, features)")
print(f"    y shape: {y.shape}")
print(f"    meta shape: {meta.shape}  (lat, lon, year, month)")
print(f"    Skipped: {skipped}")
print(f"    NaN fraction in X: {np.isnan(X).mean():.4f}")

print(f"\n  Class distribution in final dataset:")
for c in BLEACH_LABELS:
    n = (y == c).sum()
    print(f"    Class {c}: {n} ({n/len(y)*100:.1f}%)")

# ──────────────────────────────────────────────────────────────
# STEP 5: Handle NaNs and save
# ──────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 5: Saving")
print("=" * 70)

# Forward-fill NaNs within each sequence, then fill remaining with 0
# This handles occasional missing weeks in the satellite data
for i in range(X.shape[0]):
    for f in range(X.shape[2]):
        col = X[i, :, f]
        # Forward fill
        mask = np.isnan(col)
        if mask.any() and not mask.all():
            idx_arr = np.where(~mask, np.arange(len(col)), 0)
            np.maximum.accumulate(idx_arr, out=idx_arr)
            col[mask] = col[idx_arr[mask]]
        X[i, :, f] = col

# Any remaining NaNs become 0
remaining_nans = np.isnan(X).mean()
X = np.nan_to_num(X, nan=0.0)

print(f"  NaNs after forward-fill (replaced with 0): {remaining_nans:.4f}")

np.savez_compressed(
    OUTPUT_PATH,
    X=X,
    y=y,
    meta=meta,
    feature_names=FEATURE_NAMES,
    bleach_bins=BLEACH_BINS,
)

file_size = os.path.getsize(OUTPUT_PATH) / (1024**2) if os.path.exists(OUTPUT_PATH) else 0
print(f"  Saved to {OUTPUT_PATH} ({file_size:.1f} MB)")
print(f"\n  Features per timestep: {FEATURE_NAMES}")
print(f"  To load:")
print(f"    data = np.load('{OUTPUT_PATH}')")
print(f"    X, y = data['X'], data['y']")

print("\nDone!")

  After dropping non-numeric bleaching values: 34513
  Date range: 1983-01-15 00:00:00 to 2020-08-15 00:00:00
  Class distribution:
    Class 0: 19944 (57.8%)
    Class 1: 7635 (22.1%)
    Class 2: 4570 (13.2%)
    Class 3: 2364 (6.8%)

STEP 2: Opening local CoRTAD files
  Opened in 0.0s
  Time range: 1982-01-05 00:00:00 to 2022-12-27 00:00:00 (2139 weeks)

STEP 3: Extracting unique site time series
  Unique sites: 10846
  Extracting FilledSST... 

/var/folders/1x/6tp2zz4x70q7nl89j3cmqf7c0000gn/T/ipykernel_3867/1798866497.py:37: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 104. This could degrade performance. Instead, consider rechunking after loading.
  sst_ds = xr.open_dataset(SST_PATH, chunks={"time": 104})
/var/folders/1x/6tp2zz4x70q7nl89j3cmqf7c0000gn/T/ipykernel_3867/1798866497.py:38: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 104. This could degrade performance. Instead, consider rechunking after loading.
  ssta_ds = xr.open_dataset(SSTA_PATH, chunks={"time": 104})
/var/folders/1x/6tp2zz4x70q7nl89j3cmqf7c0000gn/T/ipykernel_3867/1798866497.py:39: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 104. This could degrade performance. Instead, consider rechunking after loading.
  tsa_ds = xr.open_dataset(TSA_PATH, chunks={"time": 104})


done (241.8s)
  Extracting SSTA... done (319.5s)
  Extracting SSTA_DHW... done (250.2s)
  Extracting SSTA_Frequency... done (139.1s)
  Extracting TSA... done (316.9s)
  Extracting TSA_DHW... done (141.4s)
  Extracting TSA_Frequency... done (109.3s)

  Combined feature array shape: (2139, 10846, 7)
  (time_steps, sites, features)

STEP 4: Building sequences
  Built 5000 sequences (1627 skipped)...
  Built 10000 sequences (2759 skipped)...
  Built 15000 sequences (3765 skipped)...
  Built 20000 sequences (4641 skipped)...
  Built 25000 sequences (5387 skipped)...

  Final dataset:
    X shape: (28539, 52, 7)  (samples, timesteps, features)
    y shape: (28539,)
    meta shape: (28539, 4)  (lat, lon, year, month)
    Skipped: 5974
    NaN fraction in X: 0.0001

  Class distribution in final dataset:
    Class 0: 16602 (58.2%)
    Class 1: 6418 (22.5%)
    Class 2: 3566 (12.5%)
    Class 3: 1953 (6.8%)

STEP 5: Saving
  NaNs after forward-fill (replaced with 0): 0.0001
  Saved to datasets/